# Kaggle Diabetes ML
Load -> EDA -> feature engineering -> feature selection -> baseline models -> tuning.

## Setup

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import RFE, RFECV, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
OUTPUTS_DIR = Path("outputs")
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CV_RESULTS_PATH = OUTPUTS_DIR / "cv_results.csv"

RANDOM_STATE = 42
FIG_DPI = 300
GRID_NCOLS = 4
GENDER_MAP = {"Female": 0, "Male": 1}
LOG_COLS = ["BMI", "MentHlth", "PhysHlth"]
HURDLE_COLS = ["MentHlth", "PhysHlth"]
ORDINAL_COLS = ["GenHlth", "Age", "Education", "Income"]
CLASS_PALETTE = {"No": "tab:blue", "Yes": "tab:orange"}
SOURCE_CAPTION = "Data: CDC Diabetes Health Indicators (Kaggle subset)"

SCORING = {"accuracy": "accuracy", "roc_auc": "roc_auc", "f1": "f1"}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def subplot_grid(n: int, height: float = 4) -> tuple[plt.Figure, np.ndarray]:
    """Figure with n axes on a GRID_NCOLS-wide grid, unused axes hidden."""
    fig, axes = plt.subplots(int(np.ceil(n / GRID_NCOLS)), GRID_NCOLS,
                             figsize=(4 * GRID_NCOLS, height * np.ceil(n / GRID_NCOLS)))
    axes = axes.ravel()
    for ax in axes[n:]:
        ax.set_visible(False)
    return fig, axes[:n]


def finish_fig(fig: plt.Figure, filename: str) -> None:
    """Add the source caption, save to FIGURES_DIR at FIG_DPI and show."""
    fig.text(0.01, 0.01, SOURCE_CAPTION, fontsize=8, style="italic")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(FIGURES_DIR / filename, dpi=FIG_DPI)
    plt.show()


def iqr_outliers(s: pd.Series, k: float = 1.5) -> pd.Series:
    """Boolean mask of values outside the Tukey fences."""
    q1, q3 = s.quantile([0.25, 0.75])
    return (s < q1 - k * (q3 - q1)) | (s > q3 + k * (q3 - q1))


def append_cv_results(df: pd.DataFrame, path: Path = CV_RESULTS_PATH) -> None:
    """Append rows to the results log, aligning columns with earlier experiments."""
    df = df.assign(timestamp=datetime.now().strftime("%Y-%m-%d %H:%M"))
    if path.exists():
        df = pd.concat([pd.read_csv(path), df], ignore_index=True)
    df.to_csv(path, index=False)

## Data loading

In [ ]:
def load_data() -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    """Load X_train, y_train (aligned on ID, ID as index) and X_test."""
    X_train = pd.read_csv("X_train.csv")
    y_train = pd.read_csv("y_train.csv")
    X_test = pd.read_csv("X_test.csv")

    training_data = pd.merge(X_train, y_train, on="ID", how="inner", validate="one_to_one")
    if len(training_data) != len(X_train):
        raise ValueError(f"ID mismatch: {len(X_train)} X rows vs {len(training_data)} merged rows")

    training_data = training_data.set_index("ID")
    features = [c for c in X_train.columns if c not in ("ID", "Diabetes")]
    return training_data[features], training_data["Diabetes"], X_test


X, y, X_test = load_data()
y_bin = (y == "Yes").astype(int)  # 1 = diabetes

## Outliers (Tukey fences, 1.5 x IQR)
Non-binary columns only: for 0/1 columns the IQR is 0 or 1, so the check is meaningless.

In [ ]:
non_binary_cols = [c for c in X.select_dtypes(include="number").columns if X[c].nunique() > 2]
outlier_ids = {c: X.index[iqr_outliers(X[c])].tolist() for c in non_binary_cols}
all_outlier_ids = sorted(set().union(*outlier_ids.values()))
print(f"Unique IDs flagged in >=1 column: {len(all_outlier_ids)} ({100 * len(all_outlier_ids) / len(X):.2f}% of rows)")

fig, axes = subplot_grid(len(non_binary_cols))
for ax, col in zip(axes, non_binary_cols):
    sns.boxplot(y=X[col], ax=ax, whis=1.5, flierprops={"marker": "o", "markersize": 3, "alpha": 0.4})
    ax.set(title=f"{col} (n_outliers = {len(outlier_ids[col])})", ylabel=col)
fig.suptitle("Non-binary feature distributions, Tukey fences (1.5 x IQR), training set")
finish_fig(fig, "boxplots_non_binary.png")

pd.DataFrame({
    "min": X[non_binary_cols].min(),
    "max": X[non_binary_cols].max(),
    "n_outliers": pd.Series({c: len(ids) for c, ids in outlier_ids.items()}),
}).assign(pct_outliers=lambda d: (100 * d["n_outliers"] / len(X)).round(2)).sort_values("n_outliers", ascending=False)

## Feature engineering: log1p + two-part (hurdle) encoding
- `BMI_log`, `MentHlth_log`, `PhysHlth_log`: log1p of the raw value (the raw columns are dropped)
- `MentHlth_any`, `PhysHlth_any`: 1 if at least one bad day was reported. The zeros (68 % / 57 %) are a separate group that no monotone transform can fix.

All rules are fixed (nothing is fitted), so they are leakage-free and reusable on X_test.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply fixed (non-fitted) transforms: Gender map, hurdle indicators, log1p on LOG_COLS."""
    out = df.assign(Gender=df["Gender"].map(GENDER_MAP))
    if out["Gender"].isna().any():
        raise ValueError("Unexpected Gender value")
    for col in HURDLE_COLS:
        out[f"{col}_any"] = (out[col] > 0).astype(int)
    for col in LOG_COLS:
        out[f"{col}_log"] = np.log1p(out[col])
    return out.drop(columns=LOG_COLS)


X_fe = engineer_features(X)
print(f"X_fe: {X_fe.shape} | new columns: {[c for c in X_fe.columns if c not in X.columns]}")

# % diabetes per value of the "any bad days" indicators
display(pd.concat(
    {c: y_bin.groupby(X_fe[c]).agg(n="size", pct_diabetes="mean") for c in [f"{h}_any" for h in HURDLE_COLS]},
    names=["feature", "value"],
).assign(pct_diabetes=lambda d: (100 * d["pct_diabetes"]).round(1)))

# Raw vs log1p, and log1p on the amount part only (value > 0)
pd.DataFrame({
    col: {
        "pct_zeros": 100 * (X[col] == 0).mean(),
        "skew_raw": X[col].skew(),
        "skew_log": X_fe[f"{col}_log"].skew(),
        "skew_log_positive": X_fe.loc[X[col] > 0, f"{col}_log"].skew(),
        "kurt_log_positive": X_fe.loc[X[col] > 0, f"{col}_log"].kurt(),
        "pct_outliers_raw": 100 * iqr_outliers(X[col]).mean(),
        "pct_outliers_log": 100 * iqr_outliers(X_fe[f"{col}_log"]).mean(),
        "pct_outliers_log_positive": 100 * iqr_outliers(X_fe.loc[X[col] > 0, f"{col}_log"]).mean(),
    } for col in LOG_COLS
}).T.round(2)

## EDA: distributions after feature engineering
Per-class histograms for non-binary features and per-class prevalence for binary features. Values are shown before scaling, because linear scaling changes only the axis units.

In [ ]:
binary_cols = [c for c in X_fe.columns if X_fe[c].nunique() == 2]
non_binary_fe_cols = [c for c in X_fe.columns if c not in binary_cols]
plot_df = X_fe.assign(Diabetes=y)

fig, axes = subplot_grid(len(non_binary_fe_cols), height=3.5)
for ax, col in zip(axes, non_binary_fe_cols):
    is_ordinal = col in ORDINAL_COLS
    sns.histplot(data=plot_df, x=col, hue="Diabetes", ax=ax, stat="percent", common_norm=False,
                 discrete=is_ordinal, bins="auto" if is_ordinal else 30, element="step", palette=CLASS_PALETTE)
    ax.set(title=col, xlabel=f"{col} ({'ordinal level' if is_ordinal else 'log1p value'})", ylabel="% of class")
fig.suptitle("Non-binary features after log1p / hurdle encoding, by diabetes status (training set)")
finish_fig(fig, "dist_non_binary_fe.png")

prevalence = (100 * X_fe[binary_cols].groupby(y).mean()).T
prevalence = prevalence.loc[(prevalence["Yes"] - prevalence["No"]).sort_values().index]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(binary_cols) + 1.5))
prevalence[["No", "Yes"]].plot.barh(ax=ax, color=list(CLASS_PALETTE.values()), width=0.8)
ax.set(xlabel="% of class with value = 1", ylabel="Binary feature",
       title="Binary feature prevalence by diabetes status (training set)")
ax.legend(title="Diabetes")
finish_fig(fig, "prevalence_binary_fe.png")

## Correlation
Spearman (rank-based, suits binary/ordinal data). The `_any`/`_log` pairs are correlated by construction.

In [ ]:
HIGH_CORR = 0.5

corr = X_fe.corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", vmin=-1, vmax=1, center=0, annot=True, fmt=".2f",
            annot_kws={"size": 7}, cbar_kws={"label": "Spearman rho"}, ax=ax)
ax.set(title="Spearman correlation of engineered features (training set)", xlabel="Feature", ylabel="Feature")
finish_fig(fig, "spearman_corr_fe.png")

high_pairs = corr.where(~mask).stack().rename("rho").rename_axis(["feature_a", "feature_b"]).reset_index()
high_pairs[high_pairs["rho"].abs() > HIGH_CORR].sort_values("rho", key=abs, ascending=False).round(3)

## Feature selection: filter scores (SelectKBest)
Mutual information (non-linear dependence) and chi-squared. These are univariate rankings on the full training set, used for exploration only. chi-squared is distorted for the continuous log features.

In [ ]:
mi = mutual_info_classif(X_fe, y_bin, discrete_features=~X_fe.columns.str.endswith("_log"),
                         random_state=RANDOM_STATE)
chi2_stat, chi2_p = chi2(X_fe, y_bin)  # all features are non-negative

filter_scores = pd.DataFrame({"mutual_info": mi, "chi2": chi2_stat, "chi2_p": chi2_p}, index=X_fe.columns)
filter_scores = filter_scores.assign(
    mi_rank=filter_scores["mutual_info"].rank(ascending=False).astype(int),
    chi2_rank=filter_scores["chi2"].rank(ascending=False).astype(int),
).sort_values("mutual_info", ascending=False)

fig, ax = plt.subplots(figsize=(9, 0.4 * len(filter_scores) + 1.5))
filter_scores["mutual_info"].iloc[::-1].plot.barh(ax=ax)
ax.set(xlabel="Mutual information with Diabetes (nats)", ylabel="Feature",
       title="Univariate relevance: mutual information (training set)")
finish_fig(fig, "mutual_info_fe.png")

filter_scores.round({"mutual_info": 4, "chi2": 1, "chi2_p": 4})

## Baseline models: RFE-selected vs. all 23 features
Default hyperparameters. RFECV per model with the model itself as the eliminator: its curve at k = 23 is the all-features score on the same folds. Both variants are then cross-validated for accuracy / ROC-AUC / F1, with RFE inside the pipeline (redone in every fold, so no leakage).
LogReg and SVM get a StandardScaler. SVM is `LinearSVC`, because an RBF kernel has no importances for RFE and is too slow at 53k rows.
**Runtime: about 7 min.**

In [ ]:
MODEL_MAX_ITER = 1000
SCALED_MODELS = {"LogReg", "SVM (linear)"}
BASELINE_MODELS = {
    "LogReg": LogisticRegression(max_iter=MODEL_MAX_ITER, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    "SVM (linear)": LinearSVC(max_iter=MODEL_MAX_ITER, random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "GradBoost": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "XGBoost": xgb.XGBClassifier(random_state=RANDOM_STATE, n_jobs=1, eval_metric="logloss"),
}


def base_pipeline(name: str, model) -> Pipeline:
    """Scaler (linear models only) + a fresh copy of the model."""
    steps = [("scale", StandardScaler())] if name in SCALED_MODELS else []
    return Pipeline(steps + [("model", clone(model))])


def importance_path(name: str) -> str:
    """Attribute RFE ranks features by: coefficients for linear models, importances for trees."""
    return f"named_steps.model.{'coef_' if name in SCALED_MODELS else 'feature_importances_'}"


model_rows, rfecv_curves, rfe_features = [], {}, {}
for name, model in BASELINE_MODELS.items():
    t0 = perf_counter()
    rfecv = RFECV(base_pipeline(name, model), cv=cv, scoring="accuracy",
                  importance_getter=importance_path(name), n_jobs=-1).fit(X_fe, y_bin)
    curve = rfecv_curves[name] = rfecv.cv_results_
    k_opt, best = int(rfecv.n_features_), np.argmax(curve["mean_test_score"])
    # 1-SE rule: smallest k within one std of the best score
    k_1se = int(curve["n_features"][np.argmax(curve["mean_test_score"] >= curve["mean_test_score"][best] - curve["std_test_score"][best])])
    rfe_features[name] = X_fe.columns[rfecv.support_].tolist()

    variants = {"all_23": (X_fe.shape[1], base_pipeline(name, model))}
    if k_opt < X_fe.shape[1]:
        variants["rfe"] = (k_opt, RFE(base_pipeline(name, model), n_features_to_select=k_opt,
                                      importance_getter=importance_path(name)))
    for variant, (n_feat, estimator) in variants.items():
        res = cross_validate(estimator, X_fe, y_bin, cv=cv, scoring=SCORING, n_jobs=-1)
        model_rows.append({"experiment": "baseline_rfe_vs_all", "model": name, "features": variant,
                           "n_features": n_feat, "k_1se": k_1se, "fit_time_s": round(res["fit_time"].mean(), 2)}
                          | {f"{m}_{s}": getattr(res[f"test_{m}"], s)() for m in SCORING for s in ("mean", "std")})
    print(f"{name:13s} k_opt = {k_opt:2d} | k_1se = {k_1se:2d} | {perf_counter() - t0:6.1f}s")

model_results = pd.DataFrame(model_rows)
append_cv_results(model_results)

for name, kept in rfe_features.items():
    print(f"{name:13s} RFE dropped: {[c for c in X_fe.columns if c not in kept]}")

fig, ax = plt.subplots(figsize=(11, 6))
for name, curve in rfecv_curves.items():
    line, = ax.plot(curve["n_features"], curve["mean_test_score"], marker="o", markersize=3, label=name)
    best = np.argmax(curve["mean_test_score"])
    ax.scatter(curve["n_features"][best], curve["mean_test_score"][best], s=90,
               facecolors="none", edgecolors=line.get_color(), linewidths=2)
ax.set(xlabel="Number of features retained", ylabel="CV accuracy (5-fold, mean)",
       title="RFECV curves per model, circles = optimum (training set)")
ax.legend(ncol=2)
finish_fig(fig, "rfecv_curves_all_models.png")

model_results.pivot(index="model", columns="features",
                    values=["n_features", "accuracy_mean", "roc_auc_mean", "f1_mean"]).reindex(list(BASELINE_MODELS)).round(4)

## Hyperparameter tuning (all 23 features)
GridSearchCV for small grids and RandomizedSearchCV for the boosting/forest models. Same 5 folds, refit on accuracy, with ROC-AUC and F1 also reported. The best CV score is slightly optimistic, since parameters are picked on the same folds; the Kaggle test set is the independent check.
**Runtime: about 15-30 min.**

In [ ]:
SEARCH_SPACES = {
    "LogReg": {"model__C": np.logspace(-3, 2, 6), "model__l1_ratio": [0, 1], "model__solver": ["liblinear"]},
    "DecisionTree": {"model__max_depth": [3, 5, 7, 9, 12, 15], "model__min_samples_leaf": [1, 20, 50, 100, 200],
                     "model__criterion": ["gini", "entropy"]},
    "RandomForest": {"model__n_estimators": randint(200, 601), "model__max_depth": [8, 12, 16, 20, None],
                     "model__min_samples_leaf": [1, 5, 10, 20, 50], "model__max_features": ["sqrt", "log2", 0.5]},
    "SVM (linear)": {"model__C": np.logspace(-3, 1, 5)},
    "AdaBoost": {"model__n_estimators": [50, 100, 200, 400], "model__learning_rate": [0.05, 0.1, 0.5, 1.0]},
    "GradBoost": {"model__n_estimators": randint(100, 501), "model__learning_rate": loguniform(0.01, 0.3),
                  "model__max_depth": [2, 3, 4, 5], "model__subsample": [0.7, 0.85, 1.0],
                  "model__min_samples_leaf": [1, 20, 50]},
    "XGBoost": {"model__n_estimators": randint(100, 801), "model__learning_rate": loguniform(0.01, 0.3),
                "model__max_depth": randint(2, 9), "model__subsample": uniform(0.6, 0.4),
                "model__colsample_bytree": uniform(0.5, 0.5), "model__min_child_weight": [1, 5, 10, 20]},
}
N_RANDOM_ITER = {"RandomForest": 30, "GradBoost": 30, "XGBoost": 40}  # models searched randomly

tuned_rows, best_models, best_params = [], {}, {}
for name, model in BASELINE_MODELS.items():
    t0 = perf_counter()
    common = dict(scoring=SCORING, refit="accuracy", cv=cv, n_jobs=-1)
    if name in N_RANDOM_ITER:
        search = RandomizedSearchCV(base_pipeline(name, model), SEARCH_SPACES[name], n_iter=N_RANDOM_ITER[name],
                                    random_state=RANDOM_STATE, **common)
    else:
        search = GridSearchCV(base_pipeline(name, model), SEARCH_SPACES[name], **common)
    search.fit(X_fe, y_bin)

    i, cvr = search.best_index_, search.cv_results_
    best_models[name] = search.best_estimator_
    best_params[name] = {k.removeprefix("model__"): v for k, v in search.best_params_.items()}
    tuned_rows.append({"experiment": "tuned", "model": name, "features": "all_23",
                       "n_candidates": len(cvr["params"]), "best_params": str(best_params[name])}
                      | {f"{m}_{s}": cvr[f"{s}_test_{m}"][i] for m in SCORING for s in ("mean", "std")})
    print(f"{name:13s} acc = {cvr['mean_test_accuracy'][i]:.4f} | {perf_counter() - t0:6.1f}s | {best_params[name]}")

tuned_results = pd.DataFrame(tuned_rows)
append_cv_results(tuned_results)
(OUTPUTS_DIR / "best_params.json").write_text(
    json.dumps(best_params, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

baseline_all = model_results[model_results["features"] == "all_23"].set_index("model")
tuned = tuned_results.set_index("model")
tuning_summary = pd.DataFrame({
    "baseline_acc": baseline_all["accuracy_mean"], "tuned_acc": tuned["accuracy_mean"],
    "tuned_acc_std": tuned["accuracy_std"], "baseline_auc": baseline_all["roc_auc_mean"],
    "tuned_auc": tuned["roc_auc_mean"], "tuned_f1": tuned["f1_mean"],
}).assign(delta_acc=lambda d: d["tuned_acc"] - d["baseline_acc"]).sort_values("tuned_acc", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
pos = np.arange(len(tuning_summary))
ax.bar(pos - 0.2, tuning_summary["baseline_acc"], 0.4, label="Baseline (defaults)")
ax.bar(pos + 0.2, tuning_summary["tuned_acc"], 0.4, yerr=tuning_summary["tuned_acc_std"], capsize=3, label="Tuned")
ax.set_xticks(pos, tuning_summary.index, rotation=20)
ax.set_ylim(tuning_summary[["baseline_acc", "tuned_acc"]].min().min() - 0.02, None)
ax.set(xlabel="Model", ylabel="CV accuracy (5-fold, mean)", title="Baseline vs. tuned models (training set, 23 features)")
ax.legend()
finish_fig(fig, "tuning_baseline_vs_tuned.png")

tuning_summary.round(4)

## Feature importance (best tuned tree model)

In [ ]:
def plot_feature_importance(model, feature_names: list[str], top_n: int = 20,
                            filename: str = "feature_importance.png") -> None:
    """Bar-plot the top_n feature importances of a fitted tree-based model and save it."""
    importances = model.feature_importances_
    top_n = min(top_n, len(feature_names))
    indices = np.argsort(importances)[::-1][:top_n]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(range(top_n), importances[indices])
    ax.set_xticks(range(top_n), [feature_names[i] for i in indices], rotation=45, ha="right")
    ax.set(title=f"Feature importance ({type(model).__name__}, tuned)", xlabel="Feature",
           ylabel="Importance (model-specific, normalised)")
    finish_fig(fig, filename)


best_tree = tuning_summary.index[~tuning_summary.index.isin(SCALED_MODELS)][0]
plot_feature_importance(best_models[best_tree].named_steps["model"], X_fe.columns.tolist(),
                        filename=f"feature_importance_{best_tree}_tuned.png")

## Submission (GradBoost)
GradBoost is rebuilt from `outputs/best_params.json` and refit on all training rows, so this section only needs Setup, Data loading and Feature engineering to have run. The output format is checked in code against `sample_submission.csv`.

In [ ]:
FINAL_MODEL = "GradBoost"
SUBMISSIONS_DIR = OUTPUTS_DIR / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
LABEL_MAP = {1: "Yes", 0: "No"}


def write_submission(pred: np.ndarray, X_test: pd.DataFrame, model_name: str) -> Path:
    """Validate the submission format and write it to a new timestamped file."""
    sub = pd.DataFrame({"ID": X_test["ID"].to_numpy(), "Diabetes": pd.Series(pred).map(LABEL_MAP).to_numpy()})
    expected_cols = pd.read_csv("sample_submission.csv", nrows=0).columns.tolist()
    assert sub.columns.tolist() == expected_cols == ["ID", "Diabetes"], f"Columns {sub.columns.tolist()} != {expected_cols}"
    assert len(sub) == len(X_test) and sub["ID"].equals(X_test["ID"].reset_index(drop=True)), "IDs differ from X_test"
    assert sub["ID"].is_unique, "Duplicate IDs"
    assert set(sub["Diabetes"]) <= {"Yes", "No"}, f"Invalid labels: {set(sub['Diabetes'])}"
    assert not sub.isna().any().any(), "Missing values in submission"

    path = SUBMISSIONS_DIR / f"submission_{model_name}_{datetime.now():%Y%m%d-%H%M%S}.csv"
    assert not path.exists(), f"{path} already exists"
    sub.to_csv(path, index=False)
    return path


params = json.loads((OUTPUTS_DIR / "best_params.json").read_text())[FINAL_MODEL]
final_model = GradientBoostingClassifier(random_state=RANDOM_STATE, **params).fit(X_fe, y_bin)

X_test_fe = engineer_features(X_test.set_index("ID"))[X_fe.columns]  # same columns, same order as training
pred = final_model.predict(X_test_fe)

log = pd.read_csv(CV_RESULTS_PATH)
cv_acc = log.loc[(log["experiment"] == "tuned") & (log["model"] == FINAL_MODEL), "accuracy_mean"].iloc[-1]
print(f"{FINAL_MODEL} params: {params}")
print(f"Train accuracy: {final_model.score(X_fe, y_bin):.4f} vs tuned CV accuracy: {cv_acc:.4f} (overfit check)")
print(f"Predicted Yes share in test: {100 * pred.mean():.1f}% (training set: {100 * y_bin.mean():.1f}%)")

submission_path = write_submission(pred, X_test, FINAL_MODEL)
print(f"Wrote {submission_path}")
pd.read_csv(submission_path).head()